# Example spectrograms per call type — talk figures

Pick the best-looking example of each call class (`alarm`, `warble`, `high-freq`, `stacks`) from the DAS-detected `all_calls_*.csv` corpus.

Workflow:
1. Load all_calls, take the **top-10** highest-confidence rows per class.
2. Render each top-10 set as a grid of small spectrograms, labelled with an option index.
3. Fill in `CHOSEN` with the option index you like for each class — the final cell renders only those as standalone PNG/PDF for the talk.

All panels share the same time-window length and frequency axis.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path("/mnt/home/gginosar/repos/gerbil_vocalization_analysis")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vocalization_analysis.spectrogram_viz import read_audio
from vocalization_analysis.spectrogram_viz import plot_spectrogram, COLOR_MAP
from vocalization_analysis.audio_processing_config import get_experiment_month

ALL_CALLS_DIR = Path(
    "/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/all_calls"
)
AUDIO_ROOT = Path(
    "/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio"
)
OUT_DIR = REPO_ROOT / "data" / "talk_call_examples"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CALL_TYPES = ["alarm", "warble", "high-freq", "stacks"]
CONF_COL = {
    "alarm":     "meanprob_alarm",
    "warble":    "meanprob_warble",
    "high-freq": "meanprob_high_freq",
    "stacks":    "meanprob_stacks",
}
N_OPTIONS = 10

# Spectrogram window + display parameters.
MIN_FREQ = 1_000
MAX_FREQ = 60_000
PAD_S = 0.05            # context on each side of the call
WINDOW_S = 0.30         # fixed time window shared by every panel (300 ms)
INCHES_PER_SECOND = 12.0
PANEL_H = 2.4

TITLE_FONTSIZE = 16

print(f"Saving figures to: {OUT_DIR}")

## Top-10 highest-confidence rows per class

Each call class is filtered to `event_type == <class>` and the 10 rows with the highest `meanprob_<class>` are kept. These are the options you choose from below.

In [ ]:
frames = []
for csv in sorted(ALL_CALLS_DIR.glob("all_calls_*.csv")):
    print(f"loading {csv.name}…")
    df = pd.read_csv(csv)
    df["source_csv"] = csv.name
    frames.append(df)
all_df = pd.concat(frames, ignore_index=True)
print(f"\nLoaded {len(all_df):,} rows across {len(frames)} CSVs.\n")

options: dict[str, pd.DataFrame] = {}
for cls in CALL_TYPES:
    sub = all_df[all_df["event_type"] == cls].copy()
    if sub.empty:
        print(f"[warn] no rows with event_type == {cls}")
        continue
    sub["_conf"] = sub[CONF_COL[cls]].astype(float)
    top = sub.nlargest(N_OPTIONS, "_conf").reset_index(drop=True)
    top["_dur_ms"] = (top["stop_time_file_sec"] - top["start_time_file_sec"]) * 1000
    options[cls] = top
    print(f"\n{cls}  (top {len(top)} by {CONF_COL[cls]}):")
    print(top[["_conf", "exp", "file_num", "channel",
               "start_time_file_sec", "_dur_ms"]].to_string())

## Render the 10 options per class

Each class produces one figure with a 2×5 grid. Panel titles show the **option index** (0–9), `meanprob`, `exp`, `file`, `ch`, `dur_ms`. After eyeballing, pick the index per class and fill `CHOSEN` in the final cell.

In [ ]:
def wav_path_for_row(row: pd.Series) -> Path:
    exp = int(row["exp"])
    file_num = int(row["file_num"])
    channel = int(row["channel"])
    month = get_experiment_month(exp)
    return (
        AUDIO_ROOT / month / str(exp) / "Averaged_wavs_w_annotations"
        / f"channel_{channel}_file_{file_num:03d}.wav"
    )


# Cache wav reads so re-running this cell doesn't re-read the same file twice.
_wav_cache: dict[str, tuple[int, np.ndarray]] = {}

def _get_audio(path: Path) -> tuple[int, np.ndarray]:
    key = str(path)
    if key not in _wav_cache:
        _wav_cache[key] = read_audio(path)
    return _wav_cache[key]


def _window(on: float, off: float, dur_file: float) -> tuple[float, float]:
    mid = 0.5 * (on + off)
    t0 = mid - WINDOW_S / 2
    t1 = mid + WINDOW_S / 2
    if t0 < 0:
        t0, t1 = 0.0, WINDOW_S
    if t1 > dur_file:
        t1 = dur_file
        t0 = max(0.0, t1 - WINDOW_S)
    return t0, t1


PANEL_W = INCHES_PER_SECOND * WINDOW_S
N_COLS = 5

for cls in CALL_TYPES:
    if cls not in options:
        continue
    top = options[cls]
    n = len(top)
    n_rows = int(np.ceil(n / N_COLS))
    color = COLOR_MAP.get(cls, "tab:gray")

    fig, axes = plt.subplots(
        n_rows, N_COLS,
        figsize=(PANEL_W * N_COLS, PANEL_H * n_rows),
        squeeze=False,
    )
    for opt_idx, row in top.iterrows():
        ax = axes[opt_idx // N_COLS, opt_idx % N_COLS]
        wav = wav_path_for_row(row)
        try:
            fs, x = _get_audio(wav)
        except FileNotFoundError:
            ax.text(0.5, 0.5, f"[{opt_idx}] missing wav\n{wav.name}",
                    ha="center", va="center", transform=ax.transAxes, fontsize=8)
            ax.axis("off")
            continue
        on = float(row["start_time_file_sec"])
        off = float(row["stop_time_file_sec"])
        dur_file = x.shape[0] / fs
        t0, t1 = _window(on, off, dur_file)
        a, b = int(t0 * fs), int(t1 * fs)

        mesh = plot_spectrogram(
            ax, x[a:b], fs, min_freq=MIN_FREQ, max_freq=MAX_FREQ, t_start=t0
        )
        mesh.set_rasterized(True)
        ax.set_xlim(t0, t0 + WINDOW_S)
        ax.set_title(
            f"[{opt_idx}] p={row['_conf']:.2f}  exp{int(row['exp'])} "
            f"f{int(row['file_num']):03d} ch{int(row['channel'])}  "
            f"{row['_dur_ms']:.0f} ms",
            fontsize=8, loc="left",
        )
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(labelsize=7)

    for opt_idx in range(n, n_rows * N_COLS):
        axes[opt_idx // N_COLS, opt_idx % N_COLS].axis("off")

    fig.suptitle(f"{cls} — top {n} options (pick an index for CHOSEN)",
                 color=color, fontsize=TITLE_FONTSIZE, y=1.001)
    plt.tight_layout()
    grid_path = OUT_DIR / f"options_grid__{cls}.png"
    fig.savefig(grid_path, dpi=120, bbox_inches="tight")
    print(f"Saved {grid_path}")
    plt.show()

## Render the chosen examples as standalone figures

Set the option index per class in `CHOSEN` and re-run. Each pick is saved as `example__<class>.{png,pdf}` in `OUT_DIR`.

In [ ]:
CHOSEN = {
    "alarm":     0,
    "warble":    0,
    "high-freq": 0,
    "stacks":    0,
}

for cls in CALL_TYPES:
    if cls not in options or cls not in CHOSEN:
        continue
    idx = CHOSEN[cls]
    row = options[cls].iloc[idx]
    wav = wav_path_for_row(row)
    fs, x = _get_audio(wav)
    on = float(row["start_time_file_sec"])
    off = float(row["stop_time_file_sec"])
    dur_file = x.shape[0] / fs
    t0, t1 = _window(on, off, dur_file)
    a, b = int(t0 * fs), int(t1 * fs)

    color = COLOR_MAP.get(cls, "tab:gray")
    fig, ax = plt.subplots(figsize=(PANEL_W, PANEL_H))
    mesh = plot_spectrogram(
        ax, x[a:b], fs, min_freq=MIN_FREQ, max_freq=MAX_FREQ, t_start=t0
    )
    mesh.set_rasterized(True)
    ax.set_xlim(t0, t0 + WINDOW_S)
    ax.set_title(cls, color=color, fontsize=TITLE_FONTSIZE, loc="left")
    plt.tight_layout()
    for ext in ("png", "pdf"):
        out = OUT_DIR / f"example__{cls}.{ext}"
        fig.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved example__{cls}.{{png,pdf}} → {OUT_DIR}  "
          f"(option {idx}: p={row['_conf']:.2f}, exp{int(row['exp'])} "
          f"f{int(row['file_num']):03d} ch{int(row['channel'])})")
    plt.show()